In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder.appName("Tugas6_2505060012").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/10/03 23:58:05 WARN Utils: Your hostname, mazedev-1701 resolves to a loopback address: 127.0.1.1; using 192.168.100.6 instead (on interface wlp2s0)
26/10/03 23:58:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/03 23:58:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [4]:
# Membaca tugas6_transaksi.csv
data_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
data_transaksi.printSchema()

# Membaca tugas6_produk.json
data_produk = spark.read.json("tugas6_produk.json")
data_produk.printSchema()

# Membaca tugas6_ulasan.csv
data_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)
data_ulasan.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [8]:
data_transaksi_produk = data_transaksi.join(data_produk, on="product_id", how="left")
data_transaksi_produk.printSchema()
data_gabungan = data_transaksi_produk.join(data_ulasan, on="order_id", how="left")
data_gabungan = data_gabungan.withColumn(
    "total_pendapatan",col("unit_terjual") * col("harga")
)
data_gabungan.printSchema()
data_gabungan.show()

root
 |-- product_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- total_pendapatan: long (nullable = true)

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|
+--------+----------+------------+-------------------+------+------------+-----------+------+------------

In [16]:
from pyspark.sql.functions import when

# Menambahkan kolom ada_ulasan
data_gabungan = data_gabungan.withColumn(
    "ada_ulasan",
    when(col("rating").isNull(), False)
    .otherwise(True)
)

# Menangani data null pada kolom rating
data_gabungan.na.fill(0, subset=["rating"]).show()

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+----------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|ada_ulasan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+----------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           50000|      true|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|     0|         1500000|     false|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|          100000|      true|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|          300000|      true|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|     0|          450000|     false|
|     TX5|        10|   

In [20]:
# LOAD — menyimpan hasil akhir ke HDFS dalam format Parquet, terpartisi berdasarkan kategori
!hdfs dfs -mkdir -p /user/mazedev/tugas6/hasil_etl

data_gabungan.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/mazedev/tugas6/hasil_etl"
)

print("Pipeline ETL selesai — hasil tersimpan di HDFS.")
!hdfs dfs -ls /user/mazedev/tugas6/hasil_etl

Pipeline ETL selesai — hasil tersimpan di HDFS.
Found 6 items
-rw-r--r--   3 mazedev supergroup          0 2026-10-04 00:36 /user/mazedev/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - mazedev supergroup          0 2026-10-04 00:36 /user/mazedev/tugas6/hasil_etl/kategori=Elektronik
drwxr-xr-x   - mazedev supergroup          0 2026-10-04 00:36 /user/mazedev/tugas6/hasil_etl/kategori=Fashion
drwxr-xr-x   - mazedev supergroup          0 2026-10-04 00:36 /user/mazedev/tugas6/hasil_etl/kategori=Kesehatan
drwxr-xr-x   - mazedev supergroup          0 2026-10-04 00:36 /user/mazedev/tugas6/hasil_etl/kategori=Makanan
drwxr-xr-x   - mazedev supergroup          0 2026-10-04 00:36 /user/mazedev/tugas6/hasil_etl/kategori=Rumah Tangga


In [21]:
# Verifikasi dengan membaca kembali
data_final = spark.read.parquet("hdfs://localhost:9000/user/mazedev/tugas6/hasil_etl")
print("Jumlah baris:", data_final.count())

Jumlah baris: 5000


In [26]:
data_insight = data_final.groupBy("kategori").agg(
    (avg(col("ada_ulasan").cast("double")) * 100).alias("presentase_ulasan")
).orderBy(col("presentase_ulasan").asc())

data_insight.show()

+------------+-----------------+
|    kategori|presentase_ulasan|
+------------+-----------------+
|     Makanan|68.84328358208955|
|   Kesehatan|68.88657648283039|
|     Fashion|70.14492753623188|
|Rumah Tangga| 70.5521472392638|
|  Elektronik|  70.659407138536|
+------------+-----------------+



Dari hasil tersebut, kategori makanan memiliki presentase ulasan terendah. Data ini sangat penting untuk diketahui oleh tim marketing. Melalui data ini, tim marketing dapat melakukan evaluasi, produk dengan kategori apa yang memiliki tingkat kepuasan tertinggi. Jika ada produk atau transaksi yang tidak diulas, ada kemungkinan bahwa produk tersebut tidak mendapat respon positif dari pelanggan.